# 04 - Fairness Audit (Fairlearn)

## Obiettivi didattici

1. Calcolare **metriche disaggregate per sottogruppo** (selection rate, TPR, FPR, precision) su `race` e `age`.
2. Misurare il **demographic parity gap** e l'**equalized odds gap**.
3. Discutere il trade-off tra definizioni di fairness incompatibili (Chouldechova 2017).
4. Collegare i risultati a possibili azioni cliniche/governance.

## Riferimenti

- Hardt, Price, Srebro (2016). *Equality of Opportunity in Supervised Learning*. NeurIPS 29.
- Chouldechova (2017). *Fair prediction with disparate impact*.
- Obermeyer et al. (2019). *Dissecting racial bias in an algorithm used to manage the health of populations*. Science.


In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

import joblib
from readmit_pipeline.config import MODELS_DIR
from readmit_pipeline.fairness import (
    fairness_report, aggregate_fairness_metrics,
)
from readmit_pipeline.pipeline import prepare_data
from readmit_pipeline.config import PipelineConfig


## Setup

Carichiamo il modello allenato e ricostruiamo lo split holdout test (deterministico: stesso seed = stesso split).


In [ ]:
model = joblib.load(MODELS_DIR / 'best_model.joblib')
config = PipelineConfig(random_state=42)
_, X_test, _, y_test, _, _, df_clean = prepare_data(config)
y_score = model.predict_proba(X_test)[:, 1]
y_pred = (y_score >= 0.5).astype(int)
print(f'Test size: {len(X_test)}, positivi reali: {y_test.sum()}')


## Fairness report disaggregato per sottogruppo


In [ ]:
sf = df_clean.loc[X_test.index, ['race', 'age']]
report_race = fairness_report(y_test.values, y_pred, sf['race'])
report_race


In [ ]:
report_age = fairness_report(y_test.values, y_pred, sf['age'])
report_age


## Riepilogo gap (demographic parity, equalized odds)


In [ ]:
summary = aggregate_fairness_metrics(
    y_true=y_test.values, y_pred=y_pred,
    sensitive_features_dict={'race': sf['race'], 'age': sf['age']},
)
summary


## Discussione

- Un **demographic_parity_gap > 0.10** indica che il tasso di alert differisce sensibilmente fra gruppi: e' un problema se il follow-up post-dimissione ha capacita' limitata e va distribuito equamente.
- Un **equalized_odds_gap > 0.10** indica che il modello "manca" piu' readmission per un gruppo che per un altro: implica disuguaglianza nell'efficacia clinica per sottogruppo.
- I due criteri **non sono simultaneamente raggiungibili** se i base rate (prevalenza della classe positiva) differiscono fra gruppi (impossibility theorem, Chouldechova 2017).
- Scelta consigliata in contesto di follow-up: **equalized odds** (piu' rilevante clinicamente: tutti i gruppi devono avere stessa probabilita' di essere intercettati se a rischio).
